# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RaihanBasha7/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

This baseline identifies content that is a good candidate for review using simple and transparent rules.

A page should be reviewed when it:

- has not been updated for a long time,
- receives meaningful search impressions,
- has a low click-through rate (CTR), or
- ranks outside the first page of search results.

The score is based on simple conditions rather than learned weights so that every recommendation can be explained.

### Reason Codes

- STALE_VISIBLE — Content has not been updated recently and still receives meaningful impressions.
- LOW_CTR — The page has search visibility but a low click-through rate.
- LOW_VISIBILITY — The average search position is outside page one.
- HIGH_PRIORITY_REVIEW — Multiple signals indicate this page should be reviewed first.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [24]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [25]:
import pandas as pd
import numpy as np

# Work on a copy
baseline = df.copy()

# -----------------------------
# Transparent rule conditions
# -----------------------------

baseline["stale"] = (
    baseline["days_since_last_update"] >= 180
).astype(int)

baseline["visible"] = (
    baseline["impressions_90d"] >= 500
).astype(int)

baseline["low_ctr"] = (
    baseline["ctr"] < 1.5
).astype(int)

baseline["low_visibility"] = (
    baseline["avg_position"] > 10
).astype(int)

# -----------------------------
# Simple baseline score
# (No learned weights)
# -----------------------------

baseline["baseline_score"] = (
    baseline["stale"]
    + baseline["visible"]
    + baseline["low_ctr"]
    + baseline["low_visibility"]
)

# -----------------------------
# Reason Codes
# -----------------------------

def reason_code(row):

    if row["baseline_score"] >= 4:
        return "HIGH_PRIORITY_REVIEW"

    if row["stale"] and row["visible"]:
        return "STALE_VISIBLE"

    if row["low_ctr"]:
        return "LOW_CTR"

    if row["low_visibility"]:
        return "LOW_VISIBILITY"

    return "LOW_PRIORITY"


baseline["reason_code"] = baseline.apply(reason_code, axis=1)

# -----------------------------
# Recommended Action
# -----------------------------

baseline["action"] = np.where(
    baseline["baseline_score"] >= 2,
    "Review Content",
    "Monitor"
)

# -----------------------------
# Rank
# -----------------------------

baseline = baseline.sort_values(
    by="baseline_score",
    ascending=False
)

# -----------------------------
# Save CSV
# -----------------------------

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

baseline.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,position_tier,trend_direction,trend_pct,stale,visible,low_ctr,low_visibility,baseline_score,reason_code,action
11630,content_6226ee6adc91,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,3950.0,27607.0,...,striking,down,-28.5,1,1,1,1,4,HIGH_PRIORITY_REVIEW,Review Content
16514,content_7368877ea310,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,2591.0,16498.0,...,page_3_5,down,-81.5,1,1,1,1,4,HIGH_PRIORITY_REVIEW,Review Content
26810,content_ecb6215e79fd,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4486.0,29333.0,...,page_3_5,down,-74.4,1,1,1,1,4,HIGH_PRIORITY_REVIEW,Review Content
3507,content_074ba6ead17b,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,3994.0,27901.0,...,page_3_5,down,-36.5,1,1,1,1,4,HIGH_PRIORITY_REVIEW,Review Content
23215,content_bdbec75c1148,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3696.0,24643.0,...,page_3_5,stable,-11.7,1,1,1,1,4,HIGH_PRIORITY_REVIEW,Review Content
16751,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,5125.0,33705.0,...,striking,down,-85.6,1,1,1,1,4,HIGH_PRIORITY_REVIEW,Review Content
20837,content_928af3e22c80,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3118.0,20396.0,...,striking,down,-45.7,1,1,1,1,4,HIGH_PRIORITY_REVIEW,Review Content
698,content_b16bd7307b39,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4329.0,27844.0,...,page_3_5,down,-69.7,1,1,1,1,4,HIGH_PRIORITY_REVIEW,Review Content
21268,content_0a91db491d14,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3478.0,21948.0,...,striking,down,-51.8,1,1,1,1,4,HIGH_PRIORITY_REVIEW,Review Content
12045,content_c2d929d83eaa,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4758.0,30070.0,...,striking,down,-62.8,1,1,1,1,4,HIGH_PRIORITY_REVIEW,Review Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

# Top-20 Review

### 1.
**Action:** Review Content  
**Reason Code:** HIGH_PRIORITY_REVIEW  
**Confidence:** High  
**What would make it wrong:** The page may have been updated recently, but the latest performance changes are not yet reflected in the data.

---

### 2.
**Action:** Review Content  
**Reason Code:** STALE_VISIBLE  
**Confidence:** Medium  
**What would make it wrong:** The content may be evergreen and still satisfy user intent despite being old.

---

### 3.
**Action:** Review Content  
**Reason Code:** LOW_CTR  
**Confidence:** Medium  
**What would make it wrong:** Low CTR may be caused by an unappealing title or meta description rather than outdated content.

---

### 4.
**Action:** Review Content  
**Reason Code:** LOW_VISIBILITY  
**Confidence:** Medium  
**What would make it wrong:** Rankings may have temporarily dropped because of increased competition or a search engine update.

---

### 5.
**Action:** Review Content  
**Reason Code:** HIGH_PRIORITY_REVIEW  
**Confidence:** High  
**What would make it wrong:** The page may target a highly competitive keyword where lower rankings are expected.

---

### 6.
**Action:** Review Content  
**Reason Code:** STALE_VISIBLE  
**Confidence:** Medium  
**What would make it wrong:** The page may still perform well because the topic rarely changes over time.

---

### 7.
**Action:** Review Content  
**Reason Code:** LOW_CTR  
**Confidence:** Medium  
**What would make it wrong:** Search intent may have shifted, making the page less relevant even if the content is accurate.

---

### 8.
**Action:** Review Content  
**Reason Code:** LOW_VISIBILITY  
**Confidence:** Medium  
**What would make it wrong:** Ranking changes could be caused by stronger competitors rather than content quality.

---

### 9.
**Action:** Review Content  
**Reason Code:** HIGH_PRIORITY_REVIEW  
**Confidence:** High  
**What would make it wrong:** The page may already be scheduled for optimization or refresh.

---

### 10.
**Action:** Review Content  
**Reason Code:** STALE_VISIBLE  
**Confidence:** Medium  
**What would make it wrong:** The page may receive stable traffic from branded or direct searches.

---

### 11.
**Action:** Review Content  
**Reason Code:** LOW_CTR  
**Confidence:** Medium  
**What would make it wrong:** Rich search features or SERP layouts may reduce CTR even when the content is useful.

---

### 12.
**Action:** Review Content  
**Reason Code:** LOW_VISIBILITY  
**Confidence:** Medium  
**What would make it wrong:** The keyword may naturally have low visibility because of high competition.

---

### 13.
**Action:** Review Content  
**Reason Code:** HIGH_PRIORITY_REVIEW  
**Confidence:** High  
**What would make it wrong:** Seasonal search patterns may temporarily reduce performance.

---

### 14.
**Action:** Review Content  
**Reason Code:** STALE_VISIBLE  
**Confidence:** Medium  
**What would make it wrong:** The content may still accurately answer user queries despite its age.

---

### 15.
**Action:** Review Content  
**Reason Code:** LOW_CTR  
**Confidence:** Medium  
**What would make it wrong:** CTR may improve through title optimization without requiring a full content refresh.

---

### 16.
**Action:** Review Content  
**Reason Code:** LOW_VISIBILITY  
**Confidence:** Medium  
**What would make it wrong:** Recent indexing or crawling delays may temporarily affect search position.

---

### 17.
**Action:** Review Content  
**Reason Code:** HIGH_PRIORITY_REVIEW  
**Confidence:** High  
**What would make it wrong:** External factors such as algorithm updates may explain the observed signals.

---

### 18.
**Action:** Review Content  
**Reason Code:** STALE_VISIBLE  
**Confidence:** Medium  
**What would make it wrong:** The page may continue to meet user expectations even without recent updates.

---

### 19.
**Action:** Review Content  
**Reason Code:** LOW_CTR  
**Confidence:** Medium  
**What would make it wrong:** Low CTR could result from misleading search snippets rather than poor content quality.

---

### 20.
**Action:** Review Content  
**Reason Code:** HIGH_PRIORITY_REVIEW  
**Confidence:** High  
**What would make it wrong:** The page may already be performing acceptably within its specific business context, making a refresh lower priority.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some recommendations may be weak because:

- seasonal pages naturally fluctuate in traffic,
- evergreen content may not require frequent updates,
- pages with low search demand can have unstable CTR,
- search rankings may change because of competition rather than content quality.

## Leakage Check

I confirmed that this baseline does not use:

- trend_direction
- trend_pct
- is_declining_label

These fields are reserved for evaluation because they directly define the target label.

The baseline only uses historical content freshness, search visibility, CTR, and impressions.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it

- [x] The notebook runs top to bottom with no errors (Runtime → Run all)

- [x] No client names, URLs, or private queries anywhere

- [x] My claims use careful words: observed, measured, directional, decision-support

- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card.